# End-to-end Machine Learning Report

**Team 23:** Veronica Joe (jj3470), Crystal Guo (lg3434), Shuxuan Xu (sx2412), Fangyi Lin (fl2748)

**Github Repository:** https://github.com/xsxsx-999/5243-project4


## 1. Project Overview

### 1.1 Introduction

The objective of this project is to build an end-to-end machine learning pipeline to decode loan decisions within the United States mortgage market. Using data from the Home Mortgage Disclosure Act (HMDA), we aim to perform a comprehensive analysis of the factors driving application outcomes. By engineering robust features and training predictive models, this study seeks to identify the key drivers—ranging from applicant demographics to loan characteristics—that influence whether a mortgage application is ultimately approved or denied. 

### 1.2 Data Collection

To construct a focused and computationally viable dataset, our data acquisition and sampling strategy followed a strict, predefined methodology:

* **Data Source & Geographical Scope:** We obtained historical mortgage application records from the HMDA database spanning a six-year period from **2019 to 2024**. To capture representative trends while managing data volume, the geographical scope was restricted to two of the largest and most dynamic housing markets in the U.S.: **California (CA) and Texas (TX)**.

* **Target Formulation & Filtering:** The full HMDA dataset contains multiple stages of the application lifecycle (e.g., withdrawn, incomplete). For the purpose of binary classification, we isolated records to strictly two definitive outcomes:
    * **Originated/Approved** (`action_taken = 1`), mapped to `target_y = 1`
    * **Denied** (`action_taken = 3`), mapped to `target_y = 0`

* **Stratified Sampling:** Given the massive scale of the raw dataset, we applied a **1% stratified proportional sampling fraction** (`SAMPLE_FRACTION = 0.01`) strictly within the restricted two-class subset, which guarantees that the *class proportion* of the Originated vs. Denied subset is perfectly preserved. This yielded a final analytical sample of **139,991** records (`HMDA_CA_TX_2019_2024_Master.csv`). 

### 1.3 Statistical Interpretation & Caveats:
The models developed in this study are explicitly trained to estimate the conditional probability of origination: 

$$P(\text{Originated} \mid \text{Originated or Denied})$$

They do *not* estimate the unconditional probability $P(\text{Originated})$ over all applications. All downstream interpretations in this report—including model performance metrics, feature importance rankings, and fairness analyses—are strictly conditional on the population having already reached a definitive "Originated or Denied" resolution.

## 2. Cleaning & Preprocessing the data

To ensure data integrity and prevent data leakage, our preprocessing pipeline was rigorously designed to compute all data-driven transformations (e.g., variance thresholds, imputation statistics, and outlier boundaries) strictly on the **training set**. These learned parameters were subsequently applied to the test set.

### 2.1 Datatype Formatting & Leakage Removal

* **Leakage Removal:** We systematically dropped fields that act as "post-outcome" variables—features to make sure the model only use information available *at the time the loan application is submitted*. For example, `action_taken` is the exact basis of our target variable; `denial_reason-1..4` are only populated *if* a loan is denied; and `purchaser_type` is only recorded *if* a loan is approved and subsequently sold to an institution. 

* **Type Casting:** We standardized the data structures to ensure safe downstream processing. Known continuous variables were strictly forced into numeric formats. The remaining non-target variables were explicitly cast to strings, carefully preserving missing values as pandas `<NA>`. Finally, we ensured the dependent variable `target_y` was correctly encoded as a binary integer (0/1).

### 2.2 Drop Columns
To reduce noise, prevent the curse of dimensionality, and optimize computational efficiency, we applied a strict, multi-tier pruning strategy. Features were systematically dropped based on missingness, variance, and cardinality thresholds:

#### 2.2.1 High Missingness
Any column with a missing value rate exceeding **80%** was directly dropped, as imputing such heavily sparse features would introduce severe artificial bias.

#### 2.2.2 Near-Zero Variance
Features lacking sufficient variation provide no predictive signal. For continuous columns, we dropped those with a standard deviation $\le 0.03$. For categorical columns, features were dropped if a single dominant category accounted for $\ge 98\%$ of the observations.

#### 2.2.3 High Cardinality & Tiered Filtering

To prevent sparse matrix explosion during One-Hot Encoding, we evaluated the unique value counts of categorical features. For the remaining categorical candidates, we applied a tiered threshold logic:

##### a. Auto-Drop ($>8$ unique values)
    
Highly fragmented categorical columns that exceeded 8 unique values were automatically dropped to minimize noise and multicollinearity.

##### b. Review Window ($5-8$ unique values)
    
Features containing between 5 and 8 unique values were flagged and isolated. Rather than dropping them outright, they were passed to the next pipeline stage for manual Exploratory Data Analysis (EDA). Rare or logically similar categories within these features are bucketed to improve model robustness. 

#### Details discussed as follow. !!!!!may move to appendix?

![image-2.png](attachment:image-2.png)

Based on the visual EDA of approval rates across the isolated categorical features, we executed the following consolidation strategy to maximize predictive signal while minimizing dimensionality *(Detailed distribution charts are available in the Appendix)*:

* **Retained as-is:** `covid_phase` (distinct macroeconomic trends) and `derived_ethnicity` (retained specifically to facilitate downstream Fair Lending analysis).
* **Binarized by Presence/Type:** * `co-applicant_sex`: Gender showed zero variance in approval outcomes. The true signal was simply the presence of a co-applicant. Converted to a binary `is_joint_application` (0 = Single, 1 = Joint).
  * `derived_loan_product_type`: Grouped into `First_Lien` (>80% approval) vs. `Subordinate_Lien` (~50% approval).
* **Binned by Risk/Approval Rates:**
  * `aus-1`: Grouped official systems into `Standard_AUS` (codes 1-4, ~90% approval) and mapped the rest to `Non_Standard_or_Exempt` (~60% approval).
  * `loan_purpose`: Grouped standard applications (purchase, refi) into `Standard_Purpose`, and higher-denial applications (home improvement, other) into `High_Risk_Purpose`.
  * `manufactured_home_land_property_interest`: Consolidated into `Has_Property_Interest` and `Not_Applicable_or_Exempt` based on distinct denial rate clusters.
* **Demographic Bucketing:** `applicant_age` was collapsed into broader life-stages (`<25`, `25-44`, `>44`). Anomaly placeholder values (8888, 9999) were explicitly replaced with `NaN` for safe downstream imputation.
* **Dropped:** `applicant_sex` (perfectly flat approval rates across categories; zero predictive value) and `applicant_ethnicity-1` (100% redundant with `derived_ethnicity`).

### 2.3 Feature Consolidation

To improve model interpretability and the model building process, we converted `activity_year` into a broader macroeconomic feature, `covid_phase` (<=2019: Pre-Pandemic; 2020–2021: Peak-Pandemic; >=2022: Post-Pandemic). Also, guided by EDA on approval rates, we logically grouped several sparse categorical features to reduce dimensionality and amplify predictive signal:

* `aus-1` $\rightarrow$ `aus_grouped` (Standard_AUS vs. Non_Standard_or_Exempt)
* `applicant_age` $\rightarrow$ `applicant_age_grouped` (Mapped undefined codes 8888/9999 to `NaN`)
* `derived_loan_product_type` $\rightarrow$ `loan_product_grouped` (First_Lien, Subordinate_Lien, Other)
* `manufactured_home_land_property_interest` $\rightarrow$ `manufactured_home_grouped`
* `co-applicant_sex` $\rightarrow$ `is_joint_application` (Binary)
* `loan_purpose` $\rightarrow$ `loan_purpose_grouped` (High_Risk vs. Standard)

### 2.4 Imputation
Missing values were visualized using a heatmap on a training sample to understand missingness patterns.

![image.png](attachment:image.png)

1. **Numeric Base Imputation:** We dynamically assigned imputation strategies based on the skewness of each numeric feature in the training data:
   * **Mean imputation:** Applied to features with relatively normal distributions ($|skew| \le 0.75$).
   * **Median imputation:** Applied to highly skewed features ($|skew| > 0.75$).
2. **Iterative/Regression Imputation:** For highly correlated numeric variables, we applied an Iterative Imputer (Bayesian Ridge) to preserve multivariate relationships. This was executed *after* base imputation to provide stable starting points for:
   * *Costs/Fees block:* `total_loan_costs`, `origination_charges`, `discount_points`, `lender_credits`
   * *Property/Loan block:* `property_value`, `loan_amount`, `income`
3. **Categorical Imputation:** Handled via mode imputation, ensuring pandas `<NA>` values were safely converted to `np.nan` for scikit-learn compatibility.

### 2.5 Outliers
To prevent extreme values in variables like `loan_amount` or `income` from distorting model weights, we performed **Winsorization**. Crucially. We calculated the **1st and 99th percentiles** strictly from the imputed training dataset to prevent data leakage. These boundaries were then used to cap extreme values in both the train and test sets using Pandas `.clip()`. To avoid accidentally corrupting binary or ordinal encoded integers, this capping was exclusively restricted to continuous variables.

### 2.6 One-hot-encoding
Given our geographic scope, `state_code` was directly mapped to a stable binary feature, `state_code_bin` (CA $\rightarrow$ 0, TX $\rightarrow$ 1). We explicitly set `drop='if_binary'` to reduce collinearity by representing binary categorical features as a single 0/1 column. Unseen categories in the test set were handled seamlessly via `handle_unknown='ignore'`, and the final feature space was outputted as a memory-efficient sparse matrix, fully prepared for downstream modeling.

## 3. Feature Signal Analysis (Target Correlations)

This stage quantifies which features are most associated with approval outcomes while preserving strict train-only evaluation.

All statistics are computed on finalized training artifacts from the pipeline (`X_train_imputed` with `y_train`) to ensure consistency and avoid leakage from test data.

- Train size: **111,992 rows**
- Predictors analyzed: **53** (`40` numeric, `13` categorical)
- Numeric association: **point-biserial correlation** (with Spearman as a robustness check)
- Categorical association: **Cramer's V** and **mutual information**
- Redundancy diagnostic: numeric **Spearman** correlation (`|rho| >= 0.8`)

### 3.1 Key Signal Features

The strongest numeric signals include `hoepa_status` (dominant negative association), followed by `lien_status`, `construction_method`, and several co-applicant/underwriting indicators (`is_joint_application`, `co-applicant_*_observed`, `preapproval`).

For categorical features, the most informative groups are `aus_grouped`, `loan_purpose_grouped`, `loan_product_grouped`, `derived_dwelling_category`, `manufactured_home_grouped`, and `covid_phase`.

These predictors form a strong candidate set for interpretable baseline models and provide a clear business-facing explanation of approval dynamics.

### 3.2 Multicollinearity / Redundancy Findings

We identified **13 highly correlated numeric pairs** (`|Spearman| >= 0.8`). Most of these relationships occur among co-applicant observation variables, applicant observation variables, and mortgage-structure flags.

This indicates meaningful redundancy in the raw feature space. Practically, the modeling phase can either prune near-duplicate variables for interpretability or retain all encoded information and reduce dimensionality using latent-factor methods.

## 4. Unsupervised Learning (TruncatedSVD)

Given the finalized one-hot encoding pipeline outputs a sparse design matrix, `TruncatedSVD` is the appropriate dimensionality-reduction method for this project.

Unlike dense PCA, SVD operates directly on sparse matrices, preserves computational efficiency, and avoids unnecessary densification risk.

### 4.1 Method and Controls

The unsupervised step is fit on `X_train_ohe` only, with explicit alignment checks against `y_train`. This keeps the full workflow consistent with train-only preprocessing and prevents leakage from test data.

For interpretability, component loadings are mapped back to encoded feature names (`feat_names`, or encoder-derived names when needed).

### 4.2 Results and Practical Use

SVD component selection is based on cumulative explained variance thresholds (90% / 95%) over a practical component range. The resulting latent features (`X_train_svd`, and `X_test_svd` when available) provide a compact representation for downstream modeling.

Top-loading summaries identify which encoded variables drive each latent component, supporting interpretability despite dimensionality reduction.

### 4.3 Modeling Recommendation

Use SVD-reduced features as the primary compact representation and compare them against original feature-selected baselines on validation AUC, F1, PR-AUC, and calibration.

In practice, feature-signal rankings remain the interpretability anchor, while SVD helps reduce redundancy and stabilize model training in the sparse high-dimensional space.